# Week 59, Day 2: Choosing metrics, and the noise ceiling

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w059/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Accuracy, balanced accuracy and AUC

Write balanced_acc(y, pred), the mean of the per-class accuracies, and auc(score, y), using the rank formula: the sum of the ranks of the positive scores, minus n_pos (n_pos + 1) / 2, divided by n_pos n_neg. Compare a lazy decoder (always 0) with a decoder that thresholds a noisy neural score.

In [ ]:
import numpy as np
from scipy.stats import rankdata
gm = np.random.default_rng(592)

def balanced_acc(y, pred):
    return ...

def auc(score, y):
    ranks = rankdata(score)
    n_pos, n_neg = np.sum(y == 1), np.sum(y == 0)
    return ...

y_det = (np.arange(900) < 90).astype(int)
score = 1.5 * y_det + gm.normal(size=900)                 # a neuron that fires a little more when the stimulus is present
pred_lazy = np.zeros(900); pred_real = (score > 1.0).astype(int)
acc_lazy, bal_lazy = np.mean(pred_lazy == y_det), balanced_acc(y_det, pred_lazy)
auc_real = auc(score, y_det)
print(f"lazy decoder: accuracy {acc_lazy:.3f}, balanced accuracy {bal_lazy:.3f}")
print(f"real decoder: accuracy {np.mean(pred_real == y_det):.3f}, balanced accuracy {balanced_acc(y_det, pred_real):.3f}, AUC {auc_real:.3f}")

In [ ]:
import numpy as np
def test_bal():
    "balanced_acc gives 0.5 for a decoder that always predicts the majority class"
    assert abs(balanced_acc(np.array([0, 0, 0, 1]), np.zeros(4)) - 0.5) < 1e-12
def test_auc():
    "auc matches a brute-force count of positive-negative pairs"
    assert abs(auc(np.array([0.1, 0.4, 0.35, 0.8]), np.array([0, 0, 1, 1])) - 0.75) < 1e-12
def test_story():
    "the lazy decoder has high accuracy but chance balanced accuracy, and the real decoder has AUC above 0.75"
    assert acc_lazy > 0.85 and abs(bal_lazy - 0.5) < 1e-9 and auc_real > 0.75

test_bal()
test_auc()
test_story()
print("All checks passed")

### Exercise: Right shape, wrong scale

Write r2(y, p). Then score a prediction that equals the true response multiplied by 2.5 and shifted by 3, against noisy measurements, with both correlation and R squared.

In [ ]:
def r2(y, p):
    return ...

stim = np.linspace(0, 2 * np.pi, 60)
true_resp = 5 + 3 * np.sin(stim)
measured = true_resp + 0.5 * gm.normal(size=60)
pred_scaled = 2.5 * true_resp + 3
r_scaled = np.corrcoef(measured, pred_scaled)[0, 1]
r2_scaled = r2(measured, pred_scaled)
print(f"mis-scaled prediction: correlation {r_scaled:.3f}, R squared {r2_scaled:.2f}")
print(f"correct prediction:    correlation {np.corrcoef(measured, true_resp)[0, 1]:.3f}, R squared {r2(measured, true_resp):.3f}")

In [ ]:
import numpy as np
def test_scores():
    "the mis-scaled prediction has correlation above 0.95 and negative R squared"
    assert r_scaled > 0.95 and r2_scaled < 0

test_scores()
print("All checks passed")

### Exercise: Signal power and normalised correlation

Write signal_power(R), where R has one row per repeat and one column per stimulus, using the formula above. Then, for 30 simulated neurons with different noise levels and 5 repeats each, compute the raw correlation between the true tuning (a perfect model) and the measured mean, and the normalised correlation.

In [ ]:
def signal_power(R):
    N = R.shape[0]
    return ...

def cc_norm(pred, R):
    ybar = R.mean(0)
    return np.cov(pred, ybar, bias=True)[0, 1] / np.sqrt(np.var(pred) * signal_power(R))

gc = np.random.default_rng(5930)
raw_cc, norm_cc = [], []
for neuron in range(30):
    tuning = 3 * np.sin(np.linspace(0, 2 * np.pi, 40) + gc.uniform(0, 6.3)) + gc.normal(size=40)
    R = tuning + gc.uniform(2, 6) * gc.normal(size=(5, 40))
    raw_cc.append(np.corrcoef(tuning, R.mean(0))[0, 1])
    norm_cc.append(cc_norm(tuning, R))
print(f"perfect model, raw correlation: mean {np.mean(raw_cc):.3f} (range {min(raw_cc):.2f} to {max(raw_cc):.2f})")
print(f"perfect model, normalised:      mean {np.mean(norm_cc):.3f} (range {min(norm_cc):.2f} to {max(norm_cc):.2f})")

In [ ]:
import numpy as np
def test_sp():
    "signal_power recovers the variance of the true tuning to within 25% on a large example"
    gg = np.random.default_rng(9)
    tune = gg.normal(size=400)
    R = tune + 2.0 * gg.normal(size=(10, 400))
    assert abs(signal_power(R) / np.var(tune) - 1) < 0.25
def test_norm():
    "across noisy neurons, the perfect model's raw correlation is well below 1 but its normalised correlation is close to 1"
    assert np.mean(raw_cc) < 0.85 and abs(np.mean(norm_cc) - 1) < 0.1

test_sp()
test_norm()
print("All checks passed")

### Exercise: Scoring a Poisson model

Write bits_per_spike(counts, rate), using the Poisson log-likelihood (the log k! terms cancel in the difference, so you may drop them). Score the true rates, rates with half the modulation, and overconfident rates that exaggerate the modulation fourfold.

In [ ]:
def bits_per_spike(counts, rate):
    ll = lambda lam: np.sum(counts * np.log(lam) - lam)
    base = np.full_like(rate, counts.mean())
    return ...

gb = np.random.default_rng(594)
theta = gb.uniform(0, 2 * np.pi, 3000)
rate_true = 2 * np.exp(0.8 * np.cos(theta))
counts = gb.poisson(rate_true)
mean_rate = rate_true.mean()
rate_half = mean_rate * np.exp(0.4 * np.cos(theta)) / np.mean(np.exp(0.4 * np.cos(theta)))
rate_bad = mean_rate * np.exp(3.2 * np.cos(theta)) / np.mean(np.exp(3.2 * np.cos(theta)))
bps_true, bps_half, bps_bad = (bits_per_spike(counts, r_) for r_ in (rate_true, rate_half, rate_bad))
print(f"bits per spike: true {bps_true:.3f}, half tuning {bps_half:.3f}, overconfident {bps_bad:.3f}")

In [ ]:
import numpy as np
def test_bps():
    "the true rates score above zero, and above a model with the tuning half as strong"
    assert bps_true > 0.05 and bps_true > bps_half
def test_bad():
    "a model with overconfident rates scores below zero"
    assert bps_bad < 0

test_bps()
test_bad()
print("All checks passed")